#### 1. Build a list of dataframe from loose csv files in folder and combine

In [ ]:
import glob
from xml.etree.ElementInclude import include

import pandas as pandas

csv_folder = "../data/loose_csv/*.csv"
file_paths = glob.glob(csv_folder)

print(f"Found {len(file_paths)} files")

dataframe_list = [pandas.read_csv(file) for file in file_paths]
dataframe_combined = pandas.concat(dataframe_list, ignore_index=True)
dataframe_combined = dataframe_combined.reset_index(drop=True)

dataframe_filtered_rps = dataframe_combined[dataframe_combined['Attack Name'] != 'Recon Ping Sweep']

In [ ]:
dataframe_filtered['Attack Name'].value_counts()

#### 2. Create a small dataframe for building model by grouping by Attack Name

In [ ]:
def sample_or_take_all(group):
    sample_size = min(len(group), 1000)
    return group.sample(n=sample_size, random_state=42)

small_df = dataframe_filtered.groupby('Attack Name').apply(sample_or_take_all)
small_df = small_df.reset_index(level=0)

print(small_df.shape)
print(small_df.head())

In [ ]:
small_df['Attack Name'].value_counts()

In [ ]:
small_df.columns

#### 3. Prepare Dataframe for SVM training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import numpy as numpy

# Separate target from features
targets = small_df['Attack Name']
features = small_df.select_dtypes(include=[numpy.number])

# Divide training and testing
features_train, features_test, target_train, target_test = train_test_split(features, targets, test_size=0.2, random_state=42, stratify=targets)

#### 4. Apply Scaler

In [ ]:
# Scale features
scaler = StandardScaler()
features_train_scaled = scaler.fit_transform(features_train)
features_test_scaled = scaler.transform(features_test)

#### 5. Train Model

In [ ]:
# Initialize LinearSVC
svc = SVC(kernel='poly', degree=3, gamma='scale', C=1.0, class_weight='balanced', random_state=42)

# Train model
svc.fit(features_train_scaled, target_train)

#### 6. Support Vectors

In [ ]:
# Total number of support vectors across all classes
print(f"Total Support Vectors: {svc.support_vectors_.shape[0]}")

# Number of support vectors per class
for i, class_name in enumerate(svc.classes_):
    print(f"Class '{class_name}': {svc.n_support_[i]} support vectors")

#### 7. Model Evaluation

In [ ]:
# Evaluate the model
target_prediction = svc.predict(features_test_scaled)
print(f"Accuracy score: {accuracy_score(target_test, target_prediction):.4f}")
print("Classification Report:")
print(classification_report(target_test, target_prediction, zero_division=0))

#### 8. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
cm = confusion_matrix(target_test, target_prediction)

fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=svc.classes_)
disp.plot(ax=ax, xticks_rotation='vertical', cmap=plt.cm.Blues)
plt.title("SVM Confusion Matrix")
plt.tight_layout()
plt.show()